# Robotwin VV attention plotting pipeline

The plotting implementation lives in `render_vv_attention.py`; this notebook renders `attn-exp-vv-10-6/step_024`. Within each layer, all 24 heads share one exact attention-probability colorbar from zero to that layer's finite maximum, with no percentile clipping. Different layers may use different maxima so low-amplitude early-layer structure remains visible.

Matrix orientation:

- rows: concatenated current-video Q tokens, small to large from top to bottom;
- columns: chronological video-history K tokens, small to large from left to right;
- gray cells: future K fields that were empty in the CSV;
- one small figure per `(step, layer, head)`;
- one 4×6 head grid per `(step, layer)`;
- figures are written under `<experiment>/<step_XXX>/layerN/`.

The helper uses only NumPy and Matplotlib; it does not import PyTorch.

In [ ]:
from pathlib import Path
import sys


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if all((candidate / name).is_dir() for name in ("notebooks", "data", "figures")):
            return candidate
    raise FileNotFoundError("Could not find jupyter-plot root")


PROJECT_ROOT = find_project_root()
WORKSET_NAME = "26Aug9-Robotwin-VV-attention"
HELPER_DIR = PROJECT_ROOT / "notebooks" / WORKSET_NAME
sys.path.insert(0, str(HELPER_DIR))

from render_vv_attention import (
    attention_step_summary,
    discover_attention_steps,
    layer_probability_vmax,
    load_head_matrix,
    locate_experiment,
    metadata,
    output_dir,
    render_attention_steps,
    render_layer_grid,
    render_head,
)

print(f"project : {PROJECT_ROOT}")
print(f"workset : {WORKSET_NAME}")

## Parameters

In [ ]:
ATTENTION_LINK = "attn-exp-vv-10-6"
EXPERIMENT_SLUG = None
ATTENTION_STEPS = ("step_024",)
FORMATS = ("png", "pdf")
V_MAX = None  # None = exact maximum shared by all 24 heads within each layer.
DPI = 300

# For a quick single-figure check, change these values and run the next cell.
PREVIEW_STEP = "step_024"
PREVIEW_LAYER = 0
PREVIEW_HEAD = 0

## Inspect the linked attention output

In [ ]:
EXPERIMENT_DIR, SUMMARY = locate_experiment(PROJECT_ROOT, ATTENTION_LINK, EXPERIMENT_SLUG)
AVAILABLE_STEPS = discover_attention_steps(EXPERIMENT_DIR)
if PREVIEW_STEP not in AVAILABLE_STEPS:
    raise ValueError(f"PREVIEW_STEP {PREVIEW_STEP!r} not in {AVAILABLE_STEPS}")
PREVIEW_SUMMARY = attention_step_summary(SUMMARY, PREVIEW_STEP)
EXPECTED_SHAPE, LAYERS, NUM_HEADS, ROW_BOUNDS, HISTORY_BOUNDS = metadata(
    EXPERIMENT_DIR, SUMMARY, PREVIEW_STEP
)
PREVIEW_MATRIX_DIR = EXPERIMENT_DIR / PREVIEW_STEP
FIGURES_ROOT = PROJECT_ROOT / "figures" / WORKSET_NAME

print(f"experiment : {EXPERIMENT_DIR.name}")
print(f"steps      : {AVAILABLE_STEPS}")
print(f"shape      : {EXPECTED_SHAPE}")
print(f"chunks     : {PREVIEW_SUMMARY['num_chunks']}")
print(f"timestep   : {PREVIEW_SUMMARY['scheduler_timestep']}")
print(f"layers     : {len(LAYERS)}")
print(f"heads      : {NUM_HEADS}")
print(f"Q bounds   : {ROW_BOUNDS}")
print(f"K bounds   : {HISTORY_BOUNDS}")

## Draw one figure

This cell loads all 24 heads in `PREVIEW_LAYER` to establish the shared layer colorbar, then draws one selected head. Change `PREVIEW_LAYER` and `PREVIEW_HEAD` to inspect another head.

In [ ]:
PREVIEW_HEADS = list(range(NUM_HEADS))
LAYER_MATRICES = {
    head: load_head_matrix(PREVIEW_MATRIX_DIR, EXPECTED_SHAPE, PREVIEW_LAYER, head)
    for head in PREVIEW_HEADS
}
PREVIEW_VMAX = layer_probability_vmax(LAYER_MATRICES, V_MAX)
MATRIX = LAYER_MATRICES[PREVIEW_HEAD]
SMALL_DIR = output_dir(FIGURES_ROOT, EXPERIMENT_DIR, PREVIEW_LAYER, PREVIEW_STEP)
SMALL_PATHS = render_head(
    MATRIX,
    PREVIEW_LAYER,
    PREVIEW_HEAD,
    PREVIEW_VMAX,
    ROW_BOUNDS,
    HISTORY_BOUNDS,
    SMALL_DIR,
    formats=FORMATS,
    dpi=DPI,
)
print(*SMALL_PATHS, sep="\n")

## Draw one layer in the reference style

This cell renders one complete head grid for `PREVIEW_LAYER`. The 24 heads reuse `PREVIEW_VMAX`, so the grid and every individual head figure from this layer have identical normalization and colorbar limits.

In [ ]:
LAYER_DIR = output_dir(FIGURES_ROOT, EXPERIMENT_DIR, PREVIEW_LAYER, PREVIEW_STEP)
LAYER_PATHS = render_layer_grid(
    LAYER_MATRICES,
    PREVIEW_LAYER,
    PREVIEW_HEADS,
    PREVIEW_VMAX,
    ROW_BOUNDS,
    HISTORY_BOUNDS,
    LAYER_DIR,
    formats=FORMATS,
    columns_per_row=6,
    dpi=DPI,
)
print(*LAYER_PATHS, sep="\n")

## Draw all step_024 layers

This full-render cell visits only `step_024`. It renders 720 individual head figures and 30 layer grids in each requested file format. Every layer computes one exact maximum across its 24 heads and reuses that colorbar for all individual figures and the 4×6 grid.

In [ ]:
MANIFESTS = render_attention_steps(
    PROJECT_ROOT,
    attention_link=ATTENTION_LINK,
    experiment_slug=EXPERIMENT_SLUG,
    attention_steps=ATTENTION_STEPS,
    layer_grid=True,
    formats=FORMATS,
    color_vmax=V_MAX,
    dpi=DPI,
)
MANIFESTS